# LLaMA2-7B DEW-ACC Bit-Width / Overflow Profiler

本 Notebook 使用完整 WikiText-2 test non-overlapping 2048-token workload，
對 LLaMA2-7B 全部 224 個 decoder `nn.Linear` 同時統計
`DEW_ACC_W=12..32` 的 RTL-exact overflow 行為。

- `Enew = Ew + Ea(normal)`，不使用 Psum LOD。
- 每 8 個 widths 共用同一次 Psum/Enew 計算，避免 21 次完整重算。
- `PROFILE_CONTEXTS` 可先設為 4 做速度測試，再改回 166。
- 每個 context 寫入 JSON checkpoint，可由中斷位置續跑。
- 不重新計算 PPL，不產生圖表。
- 最終只輸出 `dew_acc_bitwidth_overflow.json`。


In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm


## Imports and frozen experiment contract


In [ ]:
import gc
import json
import math
import os
import platform
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import triton
import triton.language as tl
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EXPECTED_LINEAR_LAYERS = 224
EXPECTED_CONTEXTS = 166
EXPECTED_EQUIVALENT_LOSS_TOKENS = 339_802
EXPECTED_WEIGHT_VALUES = 6_476_005_376
EXPECTED_ACTIVATION_VALUES = 387_117_481_984
PILOT_MODE = True
DEFAULT_PROFILE_CONTEXTS = 4 if PILOT_MODE else EXPECTED_CONTEXTS
PROFILE_CONTEXTS = int(
    os.getenv("DEW_PROFILE_CONTEXTS", str(DEFAULT_PROFILE_CONTEXTS))
)
CHECKPOINT_EVERY_CONTEXTS = 1
RESUME_PROFILE = True

T_SKIP_BITS = 9
T_REPLACE_BITS = 3
ACC_WIDTHS = tuple(range(12, 33))
RESULT_DIR = Path(
    os.getenv(
        "DEW_OUTPUT_DIR",
        "results/dew-acc-bitwidth-overflow/llama2-7b",
    )
)
RESULT_PATH = RESULT_DIR / "dew_acc_bitwidth_overflow.json"
CHECKPOINT_PATH = RESULT_PATH.with_name(
    f"dew_acc_bitwidth_overflow.c{PROFILE_CONTEXTS}.checkpoint.json"
)
if not 1 <= PROFILE_CONTEXTS <= EXPECTED_CONTEXTS:
    raise ValueError(
        f"PROFILE_CONTEXTS must be in [1, {EXPECTED_CONTEXTS}]."
    )


@dataclass(frozen=True)
class HybridConfig:
    block_size: int = 16
    shared_exponent_bits: int = 5
    mantissa_bits: int = 3
    rounding: str = "nearest_even"
    activation_threshold_method: str = "signed_mean_plus_sigma_k_std"
    sigma_k: float = 3.0
    max_outliers_per_block: int = 2
    topk_tie_break: str = "lowest_k_index"
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 2048
    quantize_lm_head: bool = False

    def validate(self):
        if self.block_size != 16:
            raise ValueError("This notebook is fixed to Group-16.")
        if self.shared_exponent_bits != 5:
            raise ValueError("This notebook is fixed to signed E5.")
        if self.mantissa_bits != 3:
            raise ValueError("BFP4/BiE4 require three magnitude bits.")
        if self.rounding != "nearest_even":
            raise ValueError("Unexpected rounding mode.")
        if self.max_outliers_per_block != 2:
            raise ValueError("This notebook is fixed to Top-2.")


@dataclass(frozen=True)
class DEWAConfig:
    skip_threshold_bits: int = 9
    replace_threshold_bits: int = 3
    enabled: bool = True

    def validate(self):
        if self.skip_threshold_bits <= 0:
            raise ValueError("skip_threshold_bits must be positive.")
        if self.replace_threshold_bits <= 0:
            raise ValueError("replace_threshold_bits must be positive.")


@dataclass(frozen=True)
class KernelConfig:
    group_k: int = 16
    block_m: int = 32
    block_n: int = 64
    num_warps: int = 4
    num_stages: int = 2


@dataclass(frozen=True)
class ProfileKernelConfig:
    group_k: int = 16
    block_m: int = 2
    block_n: int = 16
    widths_per_program: int = 8
    num_warps: int = 4
    num_stages: int = 2


FORMAT = HybridConfig()
KERNEL = KernelConfig()
PROFILE_KERNEL = ProfileKernelConfig()
FUNCTIONAL_DEWA = DEWAConfig(T_SKIP_BITS, T_REPLACE_BITS, True)
FORMAT.validate()
FUNCTIONAL_DEWA.validate()

EMBEDDED_AREA_BY_WIDTH = {'12': {'acc_width': 12, 'total_pe_area': 17589.197017, 'pe_combinational_area': 14748.451411, 'pe_noncombinational_area': 2840.745606, 'dew_acc_area': 2774.4192, 'dew_acc_percent_of_pe': 15.8, 'dew_acc_combinational_area': 2122.4448, 'dew_acc_noncombinational_area': 651.9744, 'area_report_sha256': '1dddb1792cefc21bb5ea18baa3861227d83c58f0ea9a1105044ae2a0bd5490b0', 'hierarchy_report_sha256': '71d7748cd0f614f912c5882f6d1f6bf5780533cfadc71c81547316339842f8d9'}, '13': {'acc_width': 13, 'total_pe_area': 17790.293028, 'pe_combinational_area': 14909.328216, 'pe_noncombinational_area': 2880.964812, 'dew_acc_area': 2932.4736, 'dew_acc_percent_of_pe': 16.5, 'dew_acc_combinational_area': 2264.976, 'dew_acc_noncombinational_area': 667.4976, 'area_report_sha256': '3a8e7a2dfe7fcd56d2b93362835a81ac8b68ab9d25bf6ab20bb66256d16da88c', 'hierarchy_report_sha256': 'c2f48570d8c246661085f596f668b013494b8e53957ecdce8302506eda87964d'}, '14': {'acc_width': 14, 'total_pe_area': 18196.71865, 'pe_combinational_area': 15300.230638, 'pe_noncombinational_area': 2896.488012, 'dew_acc_area': 3360.0673, 'dew_acc_percent_of_pe': 18.5, 'dew_acc_combinational_area': 2677.0465, 'dew_acc_noncombinational_area': 683.0208, 'area_report_sha256': '99727a5f77f1f14670bf3ab61710ee26f763b725ac71744fe21e43272c09d29d', 'hierarchy_report_sha256': '79f63cd1238c9fa42314150ef445110fcedac7240975ff035d1dc230723ef2a7'}, '15': {'acc_width': 15, 'total_pe_area': 18327.960234, 'pe_combinational_area': 15440.645028, 'pe_noncombinational_area': 2887.315207, 'dew_acc_area': 3494.8368, 'dew_acc_percent_of_pe': 19.1, 'dew_acc_combinational_area': 2796.2928, 'dew_acc_noncombinational_area': 698.544, 'area_report_sha256': 'b6cbd40b0fbc6f16d0d959718840c693ea383e90296d97cefa42d976a09772b5', 'hierarchy_report_sha256': 'a7124c00ea752c00741c7d33b07e12470c6e9f9e55ecb5e05826054187d5d123'}, '16': {'acc_width': 16, 'total_pe_area': 18649.008252, 'pe_combinational_area': 15714.417838, 'pe_noncombinational_area': 2934.590414, 'dew_acc_area': 3806.0065, 'dew_acc_percent_of_pe': 20.4, 'dew_acc_combinational_area': 3088.4113, 'dew_acc_noncombinational_area': 717.5952, 'area_report_sha256': '07a3eaabb56c24178134b9cf663ccb9037b2d587313f2042a5e50257827b22ab', 'hierarchy_report_sha256': '8d8b24824d45a3c6a1c5bd4661e45de5e8880bd800d6476a51738c52985fa631'}, '17': {'acc_width': 17, 'total_pe_area': 18810.590646, 'pe_combinational_area': 15892.229039, 'pe_noncombinational_area': 2918.361607, 'dew_acc_area': 4015.5697, 'dew_acc_percent_of_pe': 21.3, 'dew_acc_combinational_area': 3285.9793, 'dew_acc_noncombinational_area': 729.5904, 'area_report_sha256': '8ec04dea994dfe8b3ba58a4bdeed6e7cd606a89ffbed455f54c89074a77abb49', 'hierarchy_report_sha256': '4c7eeb277d00ffa353bb0b0d8522b43867f3c94455812685aafdcc23c0d4ee6f'}, '18': {'acc_width': 18, 'total_pe_area': 19096.35865, 'pe_combinational_area': 16156.123442, 'pe_noncombinational_area': 2940.235208, 'dew_acc_area': 4289.3425, 'dew_acc_percent_of_pe': 22.5, 'dew_acc_combinational_area': 3542.1121, 'dew_acc_noncombinational_area': 747.2304, 'area_report_sha256': '813d45c6fc91ebe16c8e98781dcc664476d1a7b549242fff17a62448b9ee600f', 'hierarchy_report_sha256': 'c605265fe3de5a9acbdbfcc1bfddf425e88fb7290a1d8169292d1a3be23d9977'}, '19': {'acc_width': 19, 'total_pe_area': 19243.123456, 'pe_combinational_area': 16261.963441, 'pe_noncombinational_area': 2981.160014, 'dew_acc_area': 4407.1777, 'dew_acc_percent_of_pe': 22.9, 'dew_acc_combinational_area': 3643.0129, 'dew_acc_noncombinational_area': 764.1648, 'area_report_sha256': 'ca9ff6c7f5a516a476ac19c5e7e684fe706668a0360c75d7d984cee16b9bd8ea', 'hierarchy_report_sha256': 'df38ecb8ec61ad660a696dae4589f574874b1c5f612f4212d489916facee979c'}, '20': {'acc_width': 20, 'total_pe_area': 19350.374652, 'pe_combinational_area': 16377.681844, 'pe_noncombinational_area': 2972.692808, 'dew_acc_area': 4537.7137, 'dew_acc_percent_of_pe': 23.5, 'dew_acc_combinational_area': 3760.8481, 'dew_acc_noncombinational_area': 776.8656, 'area_report_sha256': '9165e1301bc5885bb2ee1d8f1ef50edc315e92f35e14cd60feb5b788d0c3f00d', 'hierarchy_report_sha256': '0d0d71b2c5cab98a3cb77d67218312036c234ffbf42b74a9a2017c4847c88981'}, '21': {'acc_width': 21, 'total_pe_area': 19634.731474, 'pe_combinational_area': 16626.053061, 'pe_noncombinational_area': 3008.678413, 'dew_acc_area': 4794.5521, 'dew_acc_percent_of_pe': 24.4, 'dew_acc_combinational_area': 3999.3409, 'dew_acc_noncombinational_area': 795.2112, 'area_report_sha256': '197e097b070b619f549bde879bfccdaceaa678c932ebaca2340fecb3f43addd0', 'hierarchy_report_sha256': '30b6dd7d7ba2d46502e63713c1ba380ca128a09f8aeee3fbd53b141a9fda8f20'}, '22': {'acc_width': 22, 'total_pe_area': 19722.225843, 'pe_combinational_area': 16726.248236, 'pe_noncombinational_area': 2995.977607, 'dew_acc_area': 4908.8593, 'dew_acc_percent_of_pe': 24.9, 'dew_acc_combinational_area': 4101.6529, 'dew_acc_noncombinational_area': 807.2064, 'area_report_sha256': 'eb90c36f645bd0541156018dc00f9d1668292d82bdd12d46f8297832b354d81e', 'hierarchy_report_sha256': 'a6054d75163fa1493c45c70d4124cf84853d4ff443a769f174ceea636b8ec8ad'}, '23': {'acc_width': 23, 'total_pe_area': 19988.237063, 'pe_combinational_area': 16944.984248, 'pe_noncombinational_area': 3043.252814, 'dew_acc_area': 5122.6561, 'dew_acc_percent_of_pe': 25.6, 'dew_acc_combinational_area': 4296.3985, 'dew_acc_noncombinational_area': 826.2576, 'area_report_sha256': 'ca8a75e1b3e401c335113e61dd2373760da1666ce9566e598e0e56b74e03bf9d', 'hierarchy_report_sha256': '7d1f7513aa8fccd5084d6c25e87ea4b7cdd3a672830f36126f8caf21d76b8dc9'}, '24': {'acc_width': 24, 'total_pe_area': 20249.309062, 'pe_combinational_area': 17222.285056, 'pe_noncombinational_area': 3027.024007, 'dew_acc_area': 5438.0593, 'dew_acc_percent_of_pe': 26.9, 'dew_acc_combinational_area': 4599.8065, 'dew_acc_noncombinational_area': 838.2528, 'area_report_sha256': 'da1ebb1536aabfb30015f85a3ba9c79ba43aa517abd186b2229896dc22992ac1', 'hierarchy_report_sha256': 'ea905f6d473cff1d89fa36516be71bba8cbba55ffea58be7f3a00e0f30901d2b'}, '25': {'acc_width': 25, 'total_pe_area': 20460.989079, 'pe_combinational_area': 17396.568268, 'pe_noncombinational_area': 3064.420812, 'dew_acc_area': 5635.6273, 'dew_acc_percent_of_pe': 27.5, 'dew_acc_combinational_area': 4781.8513, 'dew_acc_noncombinational_area': 853.776, 'area_report_sha256': 'e50c7166298f03223b7aaf56aea45c58e74146b83688e9e3917e0f32155da772', 'hierarchy_report_sha256': 'c0ea0385eb74637df5f2b6ed164ea68c961b7b37751cb240e5db29982aa97142'}, '26': {'acc_width': 26, 'total_pe_area': 20690.309088, 'pe_combinational_area': 17605.425875, 'pe_noncombinational_area': 3084.883213, 'dew_acc_area': 5840.2513, 'dew_acc_percent_of_pe': 28.2, 'dew_acc_combinational_area': 4968.8353, 'dew_acc_noncombinational_area': 871.416, 'area_report_sha256': '9cec919e754110bdaa775464d30b98b5deb90d4bda8688c3055b016e9bb6c112', 'hierarchy_report_sha256': 'd7aa56fe5a30a0e9888eeb4235d460fb9fec467c07753e28a660dfafffb4dd7a'}, '27': {'acc_width': 27, 'total_pe_area': 21025.469114, 'pe_combinational_area': 17891.899493, 'pe_noncombinational_area': 3133.569621, 'dew_acc_area': 6130.9585, 'dew_acc_percent_of_pe': 29.2, 'dew_acc_combinational_area': 5246.1361, 'dew_acc_noncombinational_area': 884.8224, 'area_report_sha256': '9ca29522dc9a5b91dd21b4215ce3072982acbe792316f0c2f4fa741578300492', 'hierarchy_report_sha256': 'd4c1a8edd0c5d912597223295bb204e1142d0a7dde6c99248af529fe079272d5'}, '28': {'acc_width': 28, 'total_pe_area': 21081.917102, 'pe_combinational_area': 17992.094695, 'pe_noncombinational_area': 3089.822407, 'dew_acc_area': 6279.1345, 'dew_acc_percent_of_pe': 29.8, 'dew_acc_combinational_area': 5378.7889, 'dew_acc_noncombinational_area': 900.3456, 'area_report_sha256': '2d9b0a7fdd8907ce8688bc23af559e9f02cbaf3b9f374cf32e10c6955939dee7', 'hierarchy_report_sha256': '34d9d7694b79c6c1e7d197ca3b236d0fcedf7b0cfab3c40942db0c49aba1a584'}, '29': {'acc_width': 29, 'total_pe_area': 21351.456312, 'pe_combinational_area': 18246.816305, 'pe_noncombinational_area': 3104.640007, 'dew_acc_area': 6545.1457, 'dew_acc_percent_of_pe': 30.7, 'dew_acc_combinational_area': 5629.2769, 'dew_acc_noncombinational_area': 915.8688, 'area_report_sha256': '4068db03af5388e52d8c80c779f5c3e56bfc936b02cf3b29cd8f2b6af79f3f8e', 'hierarchy_report_sha256': '4a2573d4bb2c0193566f67113b7b2b3f5ff797444c36185223840ffc8a25da77'}, '30': {'acc_width': 30, 'total_pe_area': 21496.809917, 'pe_combinational_area': 18375.235511, 'pe_noncombinational_area': 3121.574407, 'dew_acc_area': 6665.8033, 'dew_acc_percent_of_pe': 31.0, 'dew_acc_combinational_area': 5733.7057, 'dew_acc_noncombinational_area': 932.0976, 'area_report_sha256': 'a7c1519d0dfc08835a463c24b08489e0012e4585484a212e60ab4c56f75c69d4', 'hierarchy_report_sha256': '0ffd9e0e606db0452eb6cad1c53cc1fa3d81de2f5a7b80a2c1bc28eaf7f81ade'}, '31': {'acc_width': 31, 'total_pe_area': 21803.040338, 'pe_combinational_area': 18625.017923, 'pe_noncombinational_area': 3178.022415, 'dew_acc_area': 6946.6321, 'dew_acc_percent_of_pe': 31.9, 'dew_acc_combinational_area': 5983.4881, 'dew_acc_noncombinational_area': 963.144, 'area_report_sha256': 'e5d8faba26e7ccfc3105dee56f1eadea078774ace1c9a7b40da936d952bee988', 'hierarchy_report_sha256': '0b0d8aaed32596ba17e3a7ec093e4bac38ad310b1f59720d84f0b1e12b37b660'}, '32': {'acc_width': 32, 'total_pe_area': 22033.771536, 'pe_combinational_area': 18881.150728, 'pe_noncombinational_area': 3152.620808, 'dew_acc_area': 7197.8257, 'dew_acc_percent_of_pe': 32.7, 'dew_acc_combinational_area': 6235.3873, 'dew_acc_noncombinational_area': 962.4384, 'area_report_sha256': 'cb83c07d7d7d627b97089acdb1e7dd0d246aaf37a502064231e61e0dc472aeab', 'hierarchy_report_sha256': '95576ee01a478d66179982a344c4d86b1f544cc8758ff441ad4492abec70e5a0'}}

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False

print(f"Widths: {ACC_WIDTHS[0]}..{ACC_WIDTHS[-1]}")
print(f"Pilot mode: {PILOT_MODE}")
print(f"Contexts: {PROFILE_CONTEXTS}/{EXPECTED_CONTEXTS}")
print(f"DEW policy: T_skip={T_SKIP_BITS}, T_replace={T_REPLACE_BITS}")
print("Profiler Enew: Ew + Ea(normal), without Psum LOD")
print("Final artifact: one JSON file")


## Hugging Face credential bootstrap


In [ ]:
import os

# This cell is intentionally idempotent for CLI re-execution in
# the same browser-attached kernel.
if os.getenv("HF_TOKEN"):
    print("HF_TOKEN is already available in this kernel environment.")
else:
    try:
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Unable to read the Colab HF_TOKEN Secret. Open this notebook "
            "in the Colab browser frontend and enable notebook access."
        ) from exc

    if not _hf_token:
        raise RuntimeError(
            "HF_TOKEN Secret is unavailable. Add it in Colab Secrets and "
            "enable notebook access before running this cell."
        )

    os.environ["HF_TOKEN"] = _hf_token
    del _hf_token
    print("HF_TOKEN loaded from Colab Secrets into kernel memory.")


## W-BFP4 and Top-2 T2A-BiE4 quantization


In [ ]:
ACTIVATION_COUNT_NAMES = (
    "total_values",
    "candidate_values",
    "encoded_outlier_values",
    "demoted_values",
    "total_blocks",
    "candidate_affected_blocks",
    "encoded_affected_blocks",
    "cap_triggered_blocks",
    "encoded_normal_only_blocks",
    "encoded_outlier_only_blocks",
    "encoded_mixed_blocks",
)
WEIGHT_COUNT_NAMES = ("total_values", "total_blocks")


def _empty_activation_counts(device):
    return torch.zeros(
        len(ACTIVATION_COUNT_NAMES), dtype=torch.int64, device=device
    )


def _empty_weight_counts(device):
    return torch.zeros(
        len(WEIGHT_COUNT_NAMES), dtype=torch.int64, device=device
    )


def _rate(numerator, denominator):
    numerator = int(numerator)
    denominator = int(denominator)
    return {
        "numerator": numerator,
        "denominator": denominator,
        "rate": None if denominator == 0 else numerator / denominator,
    }


def _counts_to_dict(counts, names):
    values = counts.detach().cpu().tolist()
    return {name: int(value) for name, value in zip(names, values)}


def _histogram_percentile(histogram, q, first_bin=0):
    if not 0.0 <= q <= 1.0:
        raise ValueError("q must be in [0, 1].")
    total = sum(histogram[first_bin:])
    if total == 0:
        return None
    rank = max(1, math.ceil(q * total))
    cumulative = 0
    for value in range(first_bin, len(histogram)):
        cumulative += histogram[value]
        if cumulative >= rank:
            return value
    raise RuntimeError("Histogram percentile closure failed.")


def _summarize_activation(
    counts, candidate_histogram, encoded_histogram, include_tail
):
    candidate_histogram = [int(value) for value in candidate_histogram]
    encoded_histogram = [int(value) for value in encoded_histogram]
    total_blocks = counts["total_blocks"]
    candidate_values = counts["candidate_values"]
    encoded_values = counts["encoded_outlier_values"]
    demoted_values = counts["demoted_values"]
    candidate_affected = counts["candidate_affected_blocks"]
    encoded_affected = counts["encoded_affected_blocks"]
    cap_triggered = counts["cap_triggered_blocks"]

    expected_length = FORMAT.block_size + 1
    if len(candidate_histogram) != expected_length:
        raise RuntimeError("Unexpected candidate histogram length.")
    if len(encoded_histogram) != expected_length:
        raise RuntimeError("Unexpected encoded histogram length.")
    if sum(candidate_histogram) != total_blocks:
        raise RuntimeError("Candidate histogram does not close over blocks.")
    if sum(encoded_histogram) != total_blocks:
        raise RuntimeError("Encoded histogram does not close over blocks.")
    if sum(
        index * value for index, value in enumerate(candidate_histogram)
    ) != candidate_values:
        raise RuntimeError("Candidate histogram does not close over values.")
    if sum(
        index * value for index, value in enumerate(encoded_histogram)
    ) != encoded_values:
        raise RuntimeError("Encoded histogram does not close over values.")
    if candidate_values - encoded_values != demoted_values:
        raise RuntimeError("Candidate/encoded/demoted value closure failed.")
    if candidate_histogram[0] != total_blocks - candidate_affected:
        raise RuntimeError("Candidate zero-bin closure failed.")
    if sum(candidate_histogram[1:]) != candidate_affected:
        raise RuntimeError("Candidate affected-block closure failed.")
    if encoded_histogram[0] != counts["encoded_normal_only_blocks"]:
        raise RuntimeError("Encoded zero-bin closure failed.")
    if sum(encoded_histogram[1:]) != encoded_affected:
        raise RuntimeError("Encoded affected-block closure failed.")
    if candidate_affected != encoded_affected:
        raise RuntimeError("A nonempty candidate set must encode an outlier.")
    if sum(
        candidate_histogram[FORMAT.max_outliers_per_block + 1 :]
    ) != cap_triggered:
        raise RuntimeError("Cap-triggered block closure failed.")
    if sum(encoded_histogram[FORMAT.max_outliers_per_block + 1 :]) != 0:
        raise RuntimeError("Encoded occupancy exceeds Top-2.")
    if (
        counts["encoded_outlier_only_blocks"]
        + counts["encoded_mixed_blocks"]
        != encoded_affected
    ):
        raise RuntimeError("Encoded affected-block partition failed.")

    max_candidate = max(
        (index for index, value in enumerate(candidate_histogram) if value),
        default=None,
    )
    max_encoded = max(
        (index for index, value in enumerate(encoded_histogram) if value),
        default=None,
    )
    percentiles = (
        ("50", 0.50),
        ("90", 0.90),
        ("95", 0.95),
        ("99", 0.99),
        ("99_9", 0.999),
        ("99_99", 0.9999),
    )
    summary = {
        "counts": counts,
        "candidate_histogram_semantics": (
            "index k = blocks with exactly k threshold candidates before capping"
        ),
        "candidate_outliers_per_block_histogram": candidate_histogram,
        "encoded_histogram_semantics": (
            "index k = blocks with exactly k encoded outliers after Top-2 capping"
        ),
        "encoded_outliers_per_block_histogram": encoded_histogram,
        "max_candidate_outliers_per_block": max_candidate,
        "max_encoded_outliers_per_block": max_encoded,
        "nearest_rank_percentiles": {
            "candidate_all_blocks": {
                f"p{label}": _histogram_percentile(candidate_histogram, q)
                for label, q in percentiles
            },
            "candidate_affected_blocks_only": {
                f"p{label}": _histogram_percentile(
                    candidate_histogram, q, first_bin=1
                )
                for label, q in percentiles
            },
            "encoded_all_blocks": {
                f"p{label}": _histogram_percentile(encoded_histogram, q)
                for label, q in percentiles
            },
        },
        "rates": {
            "candidate_value_rate": _rate(
                candidate_values, counts["total_values"]
            ),
            "encoded_outlier_value_rate": _rate(
                encoded_values, counts["total_values"]
            ),
            "demoted_value_rate": _rate(
                demoted_values, counts["total_values"]
            ),
            "demoted_fraction_of_candidates": _rate(
                demoted_values, candidate_values
            ),
            "candidate_affected_block_rate": _rate(
                candidate_affected, total_blocks
            ),
            "encoded_affected_block_rate": _rate(
                encoded_affected, total_blocks
            ),
            "cap_triggered_block_rate": _rate(cap_triggered, total_blocks),
        },
    }
    if include_tail:
        summary["candidate_tail_probabilities"] = [
            {
                "minimum_candidates": minimum,
                **_rate(sum(candidate_histogram[minimum:]), total_blocks),
            }
            for minimum in range(1, FORMAT.block_size + 1)
        ]
    return summary


@torch.no_grad()
def _signed_tensor_threshold(tensor, sigma_k, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.numel() == 0:
        raise ValueError("Cannot compute a threshold for an empty tensor.")
    if chunk_rows <= 0:
        raise ValueError("chunk_rows must be positive.")

    running_count = 0
    running_mean = torch.zeros((), dtype=torch.float64, device=flat.device)
    running_m2 = torch.zeros((), dtype=torch.float64, device=flat.device)

    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        values = flat[start:end].float()
        chunk_count = values.numel()
        chunk_var, chunk_mean = torch.var_mean(values, unbiased=False)
        chunk_mean = chunk_mean.to(torch.float64)
        chunk_var = chunk_var.to(torch.float64)

        if running_count == 0:
            running_mean = chunk_mean
            running_m2 = chunk_var * chunk_count
            running_count = chunk_count
            continue

        combined_count = running_count + chunk_count
        delta = chunk_mean - running_mean
        running_mean = running_mean + delta * (chunk_count / combined_count)
        running_m2 = (
            running_m2
            + chunk_var * chunk_count
            + delta.square() * running_count * chunk_count / combined_count
        )
        running_count = combined_count

    variance = (running_m2 / running_count).clamp_min(0.0)
    threshold = running_mean + sigma_k * torch.sqrt(variance)
    return threshold.to(torch.float32)


def _shared_exponent(max_abs, present, config):
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    scale_exponent = (
        torch.floor(torch.log2(safe_max))
        - (config.mantissa_bits - 1)
    )
    exponent_bias = (1 << (config.shared_exponent_bits - 1)) - 1
    scale_min = -exponent_bias
    scale_max = (1 << config.shared_exponent_bits) - 1 - exponent_bias
    scale_exponent = scale_exponent.clamp(scale_min, scale_max)
    max_exponent = scale_exponent + (config.mantissa_bits - 1)
    return torch.where(present, max_exponent, torch.zeros_like(max_exponent))


def _pack_bits_last_dim(mask):
    width = mask.shape[-1]
    padding = (-width) % 8
    if padding:
        mask = F.pad(mask, (0, padding), value=False)
    grouped = mask.reshape(*mask.shape[:-1], -1, 8).to(torch.int16)
    shifts = torch.arange(8, device=mask.device, dtype=torch.int16)
    shape = (1,) * (grouped.ndim - 1) + (8,)
    return (grouped << shifts.reshape(shape)).sum(dim=-1).to(torch.uint8)


def _unpack_bits_last_dim(packed, width):
    shifts = torch.arange(8, device=packed.device, dtype=torch.int16)
    shape = (1,) * packed.ndim + (8,)
    expanded = (
        (packed.to(torch.int16).unsqueeze(-1) >> shifts.reshape(shape)) & 1
    ).to(torch.bool)
    return expanded.reshape(*packed.shape[:-1], -1)[..., :width]


@torch.no_grad()
def _quantize_weight_bfp_rows(rows, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size
    if padding:
        flat = F.pad(flat, (0, padding))

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    exponent = _shared_exponent(max_abs, max_abs != 0, config)
    step = torch.pow(2.0, exponent - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(
        -mantissa_max, mantissa_max
    )
    output = (mantissa * step).reshape(flat.size(0), padded_width)
    return output[:, :width].reshape(original_shape).to(rows.dtype)


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    counts = _empty_weight_counts(weight.device)
    blocks_per_row = math.ceil(weight.size(-1) / config.block_size)

    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        rows = weight[start:end]
        rows.copy_(_quantize_weight_bfp_rows(rows, config))
        chunk_counts = torch.tensor(
            [rows.numel(), rows.size(0) * blocks_per_row],
            dtype=torch.int64,
            device=weight.device,
        )
        counts.add_(chunk_counts)
    return counts


def _select_top2_candidates(magnitude, candidate, config):
    if magnitude.shape != candidate.shape:
        raise ValueError("Magnitude and candidate masks must have the same shape.")
    if magnitude.size(-1) != config.block_size:
        raise ValueError("Top-2 selection expects G16 in the last dimension.")

    selected = torch.zeros_like(candidate)
    remaining = candidate.clone()
    positions = torch.arange(config.block_size, device=magnitude.device)
    positions = positions.view(
        *([1] * (magnitude.ndim - 1)), config.block_size
    )

    for _ in range(config.max_outliers_per_block):
        has_candidate = remaining.any(dim=-1, keepdim=True)
        score = magnitude.masked_fill(~remaining, float("-inf"))
        index = score.argmax(dim=-1, keepdim=True)
        picked = (positions == index) & has_candidate
        selected = selected | picked
        remaining = remaining & ~picked
    return selected


@torch.no_grad()
def _quantize_activation_rows_with_threshold(rows, threshold, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size

    valid = torch.ones_like(flat, dtype=torch.bool)
    if padding:
        flat = F.pad(flat, (0, padding))
        valid = F.pad(valid, (0, padding), value=False)

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    valid_blocks = valid.reshape_as(blocks)
    magnitude = blocks.abs()

    candidate = valid_blocks & (magnitude > threshold)
    outlier = _select_top2_candidates(magnitude, candidate, config)
    demoted = candidate & ~outlier
    normal = valid_blocks & ~outlier
    normal_present = normal.any(dim=-1, keepdim=True)
    outlier_present = outlier.any(dim=-1, keepdim=True)

    normal_max = torch.where(normal, magnitude, 0.0).amax(dim=-1, keepdim=True)
    outlier_max = torch.where(outlier, magnitude, 0.0).amax(dim=-1, keepdim=True)
    normal_exp = _shared_exponent(normal_max, normal_present, config)
    outlier_exp = _shared_exponent(outlier_max, outlier_present, config)
    normal_exp = torch.where(
        ~normal_present & outlier_present, outlier_exp, normal_exp
    )
    outlier_exp = torch.where(
        ~outlier_present & normal_present, normal_exp, outlier_exp
    )
    selected_exp = torch.where(outlier, outlier_exp, normal_exp)

    step = torch.pow(2.0, selected_exp - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(
        -mantissa_max, mantissa_max
    )
    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape).to(rows.dtype)
    outlier_mask = outlier.reshape(flat.size(0), padded_width)[:, :width]

    real_block = valid_blocks.any(dim=-1)
    valid_count = valid_blocks.sum(dim=-1)
    candidate_count = candidate.sum(dim=-1)
    outlier_count = outlier.sum(dim=-1)
    candidate_affected = real_block & (candidate_count > 0)
    encoded_affected = real_block & (outlier_count > 0)
    cap_triggered = real_block & (
        candidate_count > config.max_outliers_per_block
    )
    outlier_only = real_block & (outlier_count == valid_count)
    mixed = encoded_affected & ~outlier_only
    candidate_histogram = torch.bincount(
        candidate_count[real_block], minlength=config.block_size + 1
    )
    encoded_histogram = torch.bincount(
        outlier_count[real_block], minlength=config.block_size + 1
    )
    counts = torch.stack(
        (
            torch.tensor(rows.numel(), dtype=torch.int64, device=rows.device),
            candidate.sum(dtype=torch.int64),
            outlier.sum(dtype=torch.int64),
            demoted.sum(dtype=torch.int64),
            real_block.sum(dtype=torch.int64),
            candidate_affected.sum(dtype=torch.int64),
            encoded_affected.sum(dtype=torch.int64),
            cap_triggered.sum(dtype=torch.int64),
            (real_block & ~encoded_affected).sum(dtype=torch.int64),
            outlier_only.sum(dtype=torch.int64),
            mixed.sum(dtype=torch.int64),
        )
    )
    return (
        dequantized,
        outlier_mask,
        counts,
        candidate_histogram,
        encoded_histogram,
    )


@torch.no_grad()
def quantize_activation_top2(tensor, config, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    threshold = _signed_tensor_threshold(flat, config.sigma_k, chunk_rows)
    output = torch.empty_like(flat)
    packed_tags = torch.empty(
        (flat.size(0), math.ceil(width / 8)),
        dtype=torch.uint8,
        device=flat.device,
    )
    counts = _empty_activation_counts(flat.device)
    candidate_histogram = torch.zeros(
        config.block_size + 1, dtype=torch.int64, device=flat.device
    )
    encoded_histogram = torch.zeros(
        config.block_size + 1, dtype=torch.int64, device=flat.device
    )
    outlier_per_k = torch.zeros(
        width, dtype=torch.int64, device=flat.device
    )
    outlier_groups = torch.zeros(
        math.ceil(width / config.block_size),
        dtype=torch.int64,
        device=flat.device,
    )

    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        (
            quantized,
            outlier_mask,
            chunk_counts,
            chunk_candidate_histogram,
            chunk_encoded_histogram,
        ) = _quantize_activation_rows_with_threshold(
            flat[start:end], threshold, config
        )
        output[start:end] = quantized
        packed_tags[start:end] = _pack_bits_last_dim(outlier_mask)
        counts.add_(chunk_counts)
        candidate_histogram.add_(chunk_candidate_histogram)
        encoded_histogram.add_(chunk_encoded_histogram)
        outlier_per_k.add_(
            outlier_mask.sum(dim=0, dtype=torch.int64)
        )

        group_padding = (-width) % config.block_size
        grouped_mask = outlier_mask
        if group_padding:
            grouped_mask = F.pad(grouped_mask, (0, group_padding), value=False)
        outlier_groups.add_(
            grouped_mask.reshape(
                grouped_mask.size(0), -1, config.block_size
            ).any(dim=-1).sum(dim=0, dtype=torch.int64)
        )

    routing_meta = {
        "rows": flat.size(0),
        "width": width,
        "outlier_per_k": outlier_per_k,
        "outlier_groups": outlier_groups,
    }
    return (
        output.reshape_as(tensor),
        packed_tags,
        threshold,
        counts,
        candidate_histogram,
        encoded_histogram,
        routing_meta,
    )


## Frozen functional stream

此 kernel 只負責產生下一層 activation。Bit-width profiler 是獨立 sidecar，不回寫 functional output。


In [ ]:
KERNEL_STAT_NAMES = (
    "total_group_output_slots",
    "normal_initial_load",
    "zero_normal_partial",
    "normal_skip_new",
    "normal_replace_old",
    "normal_add",
    "normal_nonzero_decisions",
    "exception_routed_group_slots",
    "zero_exception_partial",
    "nonzero_exception_partial",
    "exception_fp_acc_adds",
    "final_dewa_nonzero_flushes",
)


def _empty_kernel_stats(device):
    return torch.zeros(
        len(KERNEL_STAT_NAMES), dtype=torch.int64, device=device
    )


def _named_stats(tensor, names):
    values = tensor.detach().cpu().tolist()
    return {name: int(value) for name, value in zip(names, values)}


In [ ]:
@triton.jit
def _top2_dewa_fpacc_kernel(
    x_ptr,
    x_tag_ptr,
    w_ptr,
    bias_ptr,
    y_ptr,
    stats_ptr,
    M,
    N,
    K,
    stride_xm,
    stride_xk,
    stride_xtm,
    stride_xtb,
    stride_wn,
    stride_wk,
    stride_ym,
    stride_yn,
    HAS_BIAS: tl.constexpr,
    DEWA_ENABLED: tl.constexpr,
    SKIP_THRESHOLD_BITS: tl.constexpr,
    REPLACE_THRESHOLD_BITS: tl.constexpr,
    GROUP_K: tl.constexpr,
    DOT_K: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    COLLECT_STATS: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, DOT_K)
    active_k = offs_k < GROUP_K
    tag_bytes = offs_k // 8
    tag_bits = offs_k % 8

    x_ptrs = (
        x_ptr
        + offs_m[:, None] * stride_xm
        + offs_k[None, :] * stride_xk
    )
    w_ptrs = (
        w_ptr
        + offs_n[:, None] * stride_wn
        + offs_k[None, :] * stride_wk
    )
    x_tag_ptrs = (
        x_tag_ptr
        + offs_m[:, None] * stride_xtm
        + tag_bytes[None, :] * stride_xtb
    )

    valid_m = offs_m < M
    valid_n = offs_n < N
    valid_output = valid_m[:, None] & valid_n[None, :]

    normal_acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    normal_emax = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    normal_emax_valid = (
        tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.int32) != 0
    )
    fp_acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    stat_total_slots = tl.zeros((1,), dtype=tl.int32)
    stat_normal_load = tl.zeros((1,), dtype=tl.int32)
    stat_zero_normal = tl.zeros((1,), dtype=tl.int32)
    stat_normal_skip = tl.zeros((1,), dtype=tl.int32)
    stat_normal_replace = tl.zeros((1,), dtype=tl.int32)
    stat_normal_add = tl.zeros((1,), dtype=tl.int32)
    stat_normal_decisions = tl.zeros((1,), dtype=tl.int32)
    stat_exception_routed = tl.zeros((1,), dtype=tl.int32)
    stat_zero_exception = tl.zeros((1,), dtype=tl.int32)
    stat_nonzero_exception = tl.zeros((1,), dtype=tl.int32)
    stat_exception_fp_add = tl.zeros((1,), dtype=tl.int32)

    for _ in range(0, tl.cdiv(K, GROUP_K)):
        x_block = tl.load(
            x_ptrs,
            mask=valid_m[:, None] & active_k[None, :],
            other=0.0,
        )
        w_block = tl.load(
            w_ptrs,
            mask=valid_n[:, None] & active_k[None, :],
            other=0.0,
        )
        x_tag_word = tl.load(
            x_tag_ptrs,
            mask=valid_m[:, None] & active_k[None, :],
            other=0,
        ).to(tl.int32)
        x_outlier = ((x_tag_word >> tag_bits[None, :]) & 1) != 0

        x_zero = tl.zeros_like(x_block)
        x_normal = tl.where(x_outlier, x_zero, x_block)
        x_exception = tl.where(x_outlier, x_block, x_zero)
        normal_partial = tl.dot(
            x_normal, tl.trans(w_block)
        ).to(tl.float32)
        exception_partial = tl.dot(
            x_exception, tl.trans(w_block)
        ).to(tl.float32)

        # Sticky Emax is established by the first nonzero normal partial.
        # It can move upward but never decreases after cancellation.
        new_nonzero = normal_partial != 0.0
        new_abs = tl.abs(normal_partial)
        new_exp = tl.floor(
            tl.log2(tl.where(new_nonzero, new_abs, 1.0))
        )
        normal_both = valid_output & normal_emax_valid & new_nonzero
        normal_delta = new_exp - normal_emax
        normal_skip = (
            normal_both
            & DEWA_ENABLED
            & (normal_delta <= -SKIP_THRESHOLD_BITS)
        )
        normal_replace = (
            normal_both
            & DEWA_ENABLED
            & (normal_delta >= REPLACE_THRESHOLD_BITS)
        )
        normal_add = normal_both & ~normal_skip & ~normal_replace
        normal_load = valid_output & ~normal_emax_valid & new_nonzero

        exception_nonzero = exception_partial != 0.0
        fp_acc = fp_acc + exception_partial
        normal_updated = tl.where(
            normal_load | normal_replace,
            normal_partial,
            normal_acc,
        )
        normal_acc = tl.where(
            normal_add,
            normal_acc + normal_partial,
            normal_updated,
        )
        normal_emax = tl.where(
            normal_load | normal_replace,
            new_exp,
            normal_emax,
        )
        normal_emax = tl.where(
            normal_add,
            tl.maximum(normal_emax, new_exp),
            normal_emax,
        )
        normal_emax_valid = normal_emax_valid | normal_load

        if COLLECT_STATS:
            x_group_outlier = (
                tl.sum(x_outlier.to(tl.int32), axis=1) > 0
            )
            exception_routed = (
                valid_output & x_group_outlier[:, None]
            )
            zero_exception = (
                exception_routed & ~exception_nonzero
            )
            nonzero_exception = (
                exception_routed & exception_nonzero
            )

            stat_total_slots += tl.sum(
                tl.sum(valid_output.to(tl.int32), axis=1), axis=0
            )
            stat_normal_load += tl.sum(
                tl.sum(normal_load.to(tl.int32), axis=1), axis=0
            )
            stat_zero_normal += tl.sum(
                tl.sum(
                    (
                        valid_output & ~new_nonzero
                    ).to(tl.int32),
                    axis=1,
                ),
                axis=0,
            )
            stat_normal_skip += tl.sum(
                tl.sum(normal_skip.to(tl.int32), axis=1), axis=0
            )
            stat_normal_replace += tl.sum(
                tl.sum(normal_replace.to(tl.int32), axis=1), axis=0
            )
            stat_normal_add += tl.sum(
                tl.sum(normal_add.to(tl.int32), axis=1), axis=0
            )
            stat_normal_decisions += tl.sum(
                tl.sum(normal_both.to(tl.int32), axis=1), axis=0
            )
            stat_exception_routed += tl.sum(
                tl.sum(exception_routed.to(tl.int32), axis=1),
                axis=0,
            )
            stat_zero_exception += tl.sum(
                tl.sum(zero_exception.to(tl.int32), axis=1),
                axis=0,
            )
            stat_nonzero_exception += tl.sum(
                tl.sum(nonzero_exception.to(tl.int32), axis=1),
                axis=0,
            )
            stat_exception_fp_add += tl.sum(
                tl.sum(nonzero_exception.to(tl.int32), axis=1),
                axis=0,
            )

        x_ptrs += GROUP_K * stride_xk
        w_ptrs += GROUP_K * stride_wk
        x_tag_ptrs += (GROUP_K // 8) * stride_xtb

    output = normal_acc + fp_acc
    if HAS_BIAS:
        bias = tl.load(
            bias_ptr + offs_n, mask=valid_n, other=0.0
        )
        output += bias[None, :]

    y_ptrs = (
        y_ptr
        + offs_m[:, None] * stride_ym
        + offs_n[None, :] * stride_yn
    )
    tl.store(y_ptrs, output, mask=valid_output)

    if COLLECT_STATS:
        stat_final_flush = tl.sum(
            tl.sum(
                (
                    valid_output & (normal_acc != 0.0)
                ).to(tl.int32),
                axis=1,
            ),
            axis=0,
        )
        tl.atomic_add(
            stats_ptr + 0,
            tl.sum(stat_total_slots, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 1,
            tl.sum(stat_normal_load, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 2,
            tl.sum(stat_zero_normal, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 3,
            tl.sum(stat_normal_skip, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 4,
            tl.sum(stat_normal_replace, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 5,
            tl.sum(stat_normal_add, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 6,
            tl.sum(stat_normal_decisions, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 7,
            tl.sum(stat_exception_routed, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 8,
            tl.sum(stat_zero_exception, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 9,
            tl.sum(stat_nonzero_exception, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 10,
            tl.sum(stat_exception_fp_add, axis=0).to(tl.int64),
        )
        tl.atomic_add(stats_ptr + 11, stat_final_flush.to(tl.int64))


def fused_top2_dewa_linear(
    x,
    x_packed_tags,
    weight,
    bias,
    dewa_config,
    kernel_stats,
):
    dewa_config.validate()
    original_shape = x.shape
    k = original_shape[-1]
    x_2d = x.reshape(-1, k).contiguous()
    m = x_2d.size(0)
    n = weight.size(0)

    if k != weight.size(1):
        raise ValueError("Activation and weight K dimensions differ.")
    if k % KERNEL.group_k or k % 8:
        raise ValueError("K must be divisible by Group-16 and tag width.")
    if x_packed_tags.shape != (m, k // 8):
        raise ValueError("Unexpected activation tag shape.")
    if kernel_stats.numel() != len(KERNEL_STAT_NAMES):
        raise ValueError("Unexpected kernel statistics shape.")

    output = torch.empty(
        (m, n), dtype=torch.float16, device=x.device
    )
    bias_arg = bias if bias is not None else weight
    grid = (
        triton.cdiv(m, KERNEL.block_m),
        triton.cdiv(n, KERNEL.block_n),
    )
    _top2_dewa_fpacc_kernel[grid](
        x_2d,
        x_packed_tags,
        weight,
        bias_arg,
        output,
        kernel_stats,
        M=m,
        N=n,
        K=k,
        stride_xm=x_2d.stride(0),
        stride_xk=x_2d.stride(1),
        stride_xtm=x_packed_tags.stride(0),
        stride_xtb=x_packed_tags.stride(1),
        stride_wn=weight.stride(0),
        stride_wk=weight.stride(1),
        stride_ym=output.stride(0),
        stride_yn=output.stride(1),
        HAS_BIAS=bias is not None,
        DEWA_ENABLED=dewa_config.enabled,
        SKIP_THRESHOLD_BITS=(
            dewa_config.skip_threshold_bits
        ),
        REPLACE_THRESHOLD_BITS=(
            dewa_config.replace_threshold_bits
        ),
        GROUP_K=KERNEL.group_k,
        DOT_K=32,
        BLOCK_M=KERNEL.block_m,
        BLOCK_N=KERNEL.block_n,
        COLLECT_STATS=True,
        num_warps=KERNEL.num_warps,
        num_stages=KERNEL.num_stages,
    )
    return output.reshape(*original_shape[:-1], n)


## RTL-exact PyTorch DEW-ACC profiler reference


In [ ]:
PROFILE_STAT_NAMES = (
    "total_group_output_slots",
    "normal_zero_slots",
    "normal_load_count",
    "normal_skip_count",
    "normal_replace_count",
    "normal_align_attempt_count",
    "overflow_event_count",
    "overflowed_output_count",
    "total_output_count",
    "final_nonzero_count",
)


def _empty_profile_counts(device):
    return torch.zeros(
        (len(ACC_WIDTHS), len(PROFILE_STAT_NAMES)),
        dtype=torch.int64,
        device=device,
    )


def _trunc_shift_right(values, shifts):
    values = values.to(torch.int64)
    shifts = shifts.to(torch.int64).clamp_min(0)
    magnitude = values.abs()
    shifted = torch.bitwise_right_shift(magnitude, shifts)
    return torch.where(values < 0, -shifted, shifted)


def _run_dew_width_reference(partials, exponents, width):
    if partials.shape != exponents.shape or partials.ndim != 3:
        raise ValueError("Expected matching [groups, M, N] tensors.")
    partials = partials.to(torch.int64)
    exponents = exponents.to(torch.int64)
    _, m, n = partials.shape
    device = partials.device
    valid = torch.zeros((m, n), dtype=torch.bool, device=device)
    eacc = torch.zeros((m, n), dtype=torch.int64, device=device)
    acc = torch.zeros((m, n), dtype=torch.int64, device=device)
    overflowed_once = torch.zeros_like(valid)
    counts = torch.zeros(
        len(PROFILE_STAT_NAMES), dtype=torch.int64, device=device
    )
    minimum = -(1 << (width - 1))
    maximum = (1 << (width - 1)) - 1

    for group in range(partials.size(0)):
        dp = partials[group]
        enew = exponents[group]
        nonzero = dp != 0
        load = nonzero & ~valid
        both = nonzero & valid
        delta = enew - eacc
        skip = both & (delta <= -T_SKIP_BITS)
        replace = both & (delta >= T_REPLACE_BITS)
        align = both & ~skip & ~replace

        negative_shift = (-delta).clamp_min(0)
        positive_shift = delta.clamp_min(0)
        aligned_dp = _trunc_shift_right(dp, negative_shift)
        aligned_acc = _trunc_shift_right(acc, positive_shift)
        candidate = torch.where(
            delta < 0,
            acc + aligned_dp,
            dp + aligned_acc,
        )
        overflow = align & ((candidate < minimum) | (candidate > maximum))

        next_acc = torch.where(load | replace, dp, acc)
        next_eacc = torch.where(load | replace, enew, eacc)
        next_acc = torch.where(align & ~overflow, candidate, next_acc)
        next_eacc = torch.where(
            align & ~overflow & (delta >= 0), enew, next_eacc
        )
        next_valid = valid | load
        acc = torch.where(overflow, torch.zeros_like(next_acc), next_acc)
        eacc = torch.where(overflow, torch.zeros_like(next_eacc), next_eacc)
        valid = torch.where(overflow, torch.zeros_like(next_valid), next_valid)
        overflowed_once |= overflow

        valid_outputs = m * n
        counts[0] += valid_outputs
        counts[1] += (~nonzero).sum(dtype=torch.int64)
        counts[2] += load.sum(dtype=torch.int64)
        counts[3] += skip.sum(dtype=torch.int64)
        counts[4] += replace.sum(dtype=torch.int64)
        counts[5] += align.sum(dtype=torch.int64)
        counts[6] += overflow.sum(dtype=torch.int64)

    counts[7] = overflowed_once.sum(dtype=torch.int64)
    counts[8] = m * n
    counts[9] = (valid & (acc != 0)).sum(dtype=torch.int64)
    return counts, acc, eacc, valid, overflowed_once


def profile_dew_reference(partials, exponents, widths=ACC_WIDTHS):
    rows = []
    for width in widths:
        counts, _, _, _, _ = _run_dew_width_reference(
            partials, exponents, width
        )
        rows.append(counts)
    return torch.stack(rows)


@torch.no_grad()
def reconstruct_integer_partials(x, packed_tags, weight, group_k=16):
    x = x.reshape(-1, x.shape[-1]).float()
    weight = weight.float()
    tags = _unpack_bits_last_dim(packed_tags, x.size(1))
    partials = []
    exponents = []
    for start in range(0, x.size(1), group_k):
        end = start + group_k
        x_group = x[:, start:end]
        w_group = weight[:, start:end]
        normal = torch.where(tags[:, start:end], 0.0, x_group)
        x_max = normal.abs().amax(dim=1)
        w_max = w_group.abs().amax(dim=1)
        x_exp = torch.floor(
            torch.log2(torch.where(x_max > 0, x_max, 1.0))
        ).to(torch.int64) - (FORMAT.mantissa_bits - 1)
        w_exp = torch.floor(
            torch.log2(torch.where(w_max > 0, w_max, 1.0))
        ).to(torch.int64) - (FORMAT.mantissa_bits - 1)
        x_mantissa = torch.round(
            normal * torch.pow(2.0, -x_exp.float()).unsqueeze(1)
        ).to(torch.int64)
        w_mantissa = torch.round(
            w_group * torch.pow(2.0, -w_exp.float()).unsqueeze(1)
        ).to(torch.int64)
        partials.append(
            (
                x_mantissa.float()
                @ w_mantissa.float().transpose(0, 1)
            ).to(torch.int64)
        )
        exponents.append(x_exp[:, None] + w_exp[None, :])
    return torch.stack(partials), torch.stack(exponents)


## Triton W12-W32 sidecar profiler


In [ ]:
@triton.jit
def _dew_width_profile_kernel(
    x_ptr,
    x_tag_ptr,
    w_ptr,
    stats_ptr,
    M,
    N,
    K,
    stride_xm,
    stride_xk,
    stride_xtm,
    stride_xtb,
    stride_wn,
    stride_wk,
    WIDTH_MIN: tl.constexpr,
    WIDTH_MAX: tl.constexpr,
    GROUP_K: tl.constexpr,
    DOT_K: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_OUTPUTS: tl.constexpr,
    WIDTHS_PER_PROGRAM: tl.constexpr,
    NUM_STATS: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    pid_width_block = tl.program_id(2)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, DOT_K)
    active_k = offs_k < GROUP_K
    offs_width = (
        pid_width_block * WIDTHS_PER_PROGRAM
        + tl.arange(0, WIDTHS_PER_PROGRAM)
    )
    acc_width = WIDTH_MIN + offs_width
    valid_width = acc_width <= WIDTH_MAX
    valid_m = offs_m < M
    valid_n = offs_n < N
    valid_output_2d = valid_m[:, None] & valid_n[None, :]
    valid_output_flat = tl.reshape(
        valid_output_2d, (BLOCK_OUTPUTS,)
    )
    valid_output = valid_output_flat[:, None] & valid_width[None, :]

    x_ptrs = x_ptr + offs_m[:, None] * stride_xm + offs_k[None, :] * stride_xk
    w_ptrs = w_ptr + offs_n[:, None] * stride_wn + offs_k[None, :] * stride_wk
    tag_bytes = offs_k // 8
    tag_bits = offs_k % 8
    tag_ptrs = (
        x_tag_ptr
        + offs_m[:, None] * stride_xtm
        + tag_bytes[None, :] * stride_xtb
    )

    acc = tl.zeros((BLOCK_OUTPUTS, WIDTHS_PER_PROGRAM), dtype=tl.int64)
    eacc = tl.zeros((BLOCK_OUTPUTS, WIDTHS_PER_PROGRAM), dtype=tl.int32)
    acc_valid = tl.zeros((BLOCK_OUTPUTS, WIDTHS_PER_PROGRAM), dtype=tl.int1)
    overflowed_once = tl.zeros((BLOCK_OUTPUTS, WIDTHS_PER_PROGRAM), dtype=tl.int1)

    stat_total_slots = tl.zeros((WIDTHS_PER_PROGRAM,), dtype=tl.int64)
    stat_zero = tl.zeros((WIDTHS_PER_PROGRAM,), dtype=tl.int64)
    stat_load = tl.zeros((WIDTHS_PER_PROGRAM,), dtype=tl.int64)
    stat_skip = tl.zeros((WIDTHS_PER_PROGRAM,), dtype=tl.int64)
    stat_replace = tl.zeros((WIDTHS_PER_PROGRAM,), dtype=tl.int64)
    stat_align = tl.zeros((WIDTHS_PER_PROGRAM,), dtype=tl.int64)
    stat_overflow = tl.zeros((WIDTHS_PER_PROGRAM,), dtype=tl.int64)

    limit = tl.full((1,), 1, tl.int64) << (acc_width - 1)
    minimum = -limit
    maximum = limit - 1

    for _ in range(0, tl.cdiv(K, GROUP_K)):
        x_block = tl.load(
            x_ptrs,
            mask=valid_m[:, None] & active_k[None, :],
            other=0.0,
        )
        w_block = tl.load(
            w_ptrs,
            mask=valid_n[:, None] & active_k[None, :],
            other=0.0,
        )
        tag_word = tl.load(
            tag_ptrs,
            mask=valid_m[:, None] & active_k[None, :],
            other=0,
        ).to(tl.int32)
        outlier = ((tag_word >> tag_bits[None, :]) & 1) != 0
        x_normal = tl.where(outlier, 0.0, x_block).to(tl.float32)
        w_float = w_block.to(tl.float32)

        x_max = tl.max(tl.abs(x_normal), axis=1)
        w_max = tl.max(tl.abs(w_float), axis=1)
        x_exp = tl.floor(tl.log2(tl.where(x_max > 0.0, x_max, 1.0))).to(tl.int32) - 2
        w_exp = tl.floor(tl.log2(tl.where(w_max > 0.0, w_max, 1.0))).to(tl.int32) - 2
        x_inv_step = tl.exp2(-x_exp.to(tl.float32))
        w_inv_step = tl.exp2(-w_exp.to(tl.float32))
        x_mantissa = (x_normal * x_inv_step[:, None]).to(tl.int8)
        w_mantissa = (w_float * w_inv_step[:, None]).to(tl.int8)
        dp_2d = tl.dot(
            x_mantissa,
            tl.trans(w_mantissa),
            out_dtype=tl.int32,
        ).to(tl.int64)
        enew_2d = x_exp[:, None] + w_exp[None, :]

        dp = tl.reshape(dp_2d, (BLOCK_OUTPUTS,))[:, None]
        enew = tl.reshape(enew_2d, (BLOCK_OUTPUTS,))[:, None]
        nonzero = dp != 0
        load = valid_output & nonzero & ~acc_valid
        both = valid_output & nonzero & acc_valid
        delta = enew - eacc
        skip = both & (delta <= -9)
        replace = both & (delta >= 3)
        align = both & ~skip & ~replace

        negative_shift = tl.maximum(-delta, 0)
        positive_shift = tl.maximum(delta, 0)
        dp_magnitude = tl.where(dp < 0, -dp, dp)
        acc_magnitude = tl.where(acc < 0, -acc, acc)
        shifted_dp_magnitude = dp_magnitude >> negative_shift
        shifted_acc_magnitude = acc_magnitude >> positive_shift
        shifted_dp = tl.where(dp < 0, -shifted_dp_magnitude, shifted_dp_magnitude)
        shifted_acc = tl.where(acc < 0, -shifted_acc_magnitude, shifted_acc_magnitude)
        candidate = tl.where(delta < 0, acc + shifted_dp, dp + shifted_acc)
        overflow = align & ((candidate < minimum) | (candidate > maximum))

        next_acc = tl.where(load | replace, dp, acc)
        next_eacc = tl.where(load | replace, enew, eacc)
        next_acc = tl.where(align & ~overflow, candidate, next_acc)
        next_eacc = tl.where(
            align & ~overflow & (delta >= 0), enew, next_eacc
        )
        next_valid = acc_valid | load
        acc = tl.where(overflow, 0, next_acc)
        eacc = tl.where(overflow, 0, next_eacc)
        acc_valid = tl.where(overflow, False, next_valid)
        overflowed_once = overflowed_once | overflow

        stat_total_slots += tl.sum(valid_output.to(tl.int64), axis=0)
        stat_zero += tl.sum((valid_output & ~nonzero).to(tl.int64), axis=0)
        stat_load += tl.sum(load.to(tl.int64), axis=0)
        stat_skip += tl.sum(skip.to(tl.int64), axis=0)
        stat_replace += tl.sum(replace.to(tl.int64), axis=0)
        stat_align += tl.sum(align.to(tl.int64), axis=0)
        stat_overflow += tl.sum(overflow.to(tl.int64), axis=0)

        x_ptrs += GROUP_K * stride_xk
        w_ptrs += GROUP_K * stride_wk
        tag_ptrs += (GROUP_K // 8) * stride_xtb

    base = offs_width * NUM_STATS
    final_overflowed = tl.sum(
        (valid_output & overflowed_once).to(tl.int64), axis=0
    )
    total_outputs = tl.sum(valid_output.to(tl.int64), axis=0)
    final_nonzero = tl.sum(
        (valid_output & acc_valid & (acc != 0)).to(tl.int64), axis=0
    )
    tl.atomic_add(stats_ptr + base + 0, stat_total_slots, mask=valid_width)
    tl.atomic_add(stats_ptr + base + 1, stat_zero, mask=valid_width)
    tl.atomic_add(stats_ptr + base + 2, stat_load, mask=valid_width)
    tl.atomic_add(stats_ptr + base + 3, stat_skip, mask=valid_width)
    tl.atomic_add(stats_ptr + base + 4, stat_replace, mask=valid_width)
    tl.atomic_add(stats_ptr + base + 5, stat_align, mask=valid_width)
    tl.atomic_add(stats_ptr + base + 6, stat_overflow, mask=valid_width)
    tl.atomic_add(stats_ptr + base + 7, final_overflowed, mask=valid_width)
    tl.atomic_add(stats_ptr + base + 8, total_outputs, mask=valid_width)
    tl.atomic_add(stats_ptr + base + 9, final_nonzero, mask=valid_width)


def profile_dew_widths(x, packed_tags, weight, profile_counts):
    original_shape = x.shape
    k = original_shape[-1]
    x_2d = x.reshape(-1, k).contiguous()
    m = x_2d.size(0)
    n = weight.size(0)
    if k != weight.size(1) or k % PROFILE_KERNEL.group_k:
        raise ValueError("Profiler requires matching Group-16 K.")
    if packed_tags.shape != (m, k // 8):
        raise ValueError("Unexpected packed tag shape.")
    if profile_counts.shape != (
        len(ACC_WIDTHS), len(PROFILE_STAT_NAMES)
    ):
        raise ValueError("Unexpected profiler counter shape.")
    grid = (
        triton.cdiv(m, PROFILE_KERNEL.block_m),
        triton.cdiv(n, PROFILE_KERNEL.block_n),
        triton.cdiv(
            len(ACC_WIDTHS), PROFILE_KERNEL.widths_per_program
        ),
    )
    _dew_width_profile_kernel[grid](
        x_2d,
        packed_tags,
        weight,
        profile_counts,
        M=m,
        N=n,
        K=k,
        stride_xm=x_2d.stride(0),
        stride_xk=x_2d.stride(1),
        stride_xtm=packed_tags.stride(0),
        stride_xtb=packed_tags.stride(1),
        stride_wn=weight.stride(0),
        stride_wk=weight.stride(1),
        WIDTH_MIN=ACC_WIDTHS[0],
        WIDTH_MAX=ACC_WIDTHS[-1],
        GROUP_K=PROFILE_KERNEL.group_k,
        DOT_K=32,
        BLOCK_M=PROFILE_KERNEL.block_m,
        BLOCK_N=PROFILE_KERNEL.block_n,
        BLOCK_OUTPUTS=(
            PROFILE_KERNEL.block_m * PROFILE_KERNEL.block_n
        ),
        WIDTHS_PER_PROGRAM=PROFILE_KERNEL.widths_per_program,
        NUM_STATS=len(PROFILE_STAT_NAMES),
        num_warps=PROFILE_KERNEL.num_warps,
        num_stages=PROFILE_KERNEL.num_stages,
    )


## Profiled Linear wrapper and raw-count aggregation


In [ ]:
class ProfiledTop2DEWLinear(nn.Module):
    def __init__(self, linear, layer_name, weight_counts):
        super().__init__()
        self.linear = linear
        self.layer_name = layer_name
        self.weight_counts = _counts_to_dict(
            weight_counts, WEIGHT_COUNT_NAMES
        )
        device = linear.weight.device
        self.register_buffer(
            "_activation_counts",
            _empty_activation_counts(device),
            persistent=False,
        )
        self.register_buffer(
            "_profile_counts",
            _empty_profile_counts(device),
            persistent=False,
        )
        self.register_buffer(
            "_functional_kernel_stats",
            _empty_kernel_stats(device),
            persistent=False,
        )
        self.register_buffer(
            "_forward_calls",
            torch.zeros((), dtype=torch.int64, device=device),
            persistent=False,
        )

    def reset_runtime_stats(self):
        self._activation_counts.zero_()
        self._profile_counts.zero_()
        self._functional_kernel_stats.zero_()
        self._forward_calls.zero_()

    def checkpoint_state(self):
        return {
            "activation_counts": self._activation_counts.cpu().tolist(),
            "profile_counts": self._profile_counts.cpu().tolist(),
            "functional_kernel_stats": (
                self._functional_kernel_stats.cpu().tolist()
            ),
            "forward_calls": int(self._forward_calls.cpu()),
        }

    def restore_checkpoint_state(self, state):
        device = self._profile_counts.device
        self._activation_counts.copy_(
            torch.tensor(
                state["activation_counts"],
                dtype=torch.int64,
                device=device,
            )
        )
        self._profile_counts.copy_(
            torch.tensor(
                state["profile_counts"],
                dtype=torch.int64,
                device=device,
            )
        )
        self._functional_kernel_stats.copy_(
            torch.tensor(
                state["functional_kernel_stats"],
                dtype=torch.int64,
                device=device,
            )
        )
        self._forward_calls.fill_(int(state["forward_calls"]))

    def forward(self, x):
        (
            x_quantized,
            packed_tags,
            _,
            activation_counts,
            _,
            _,
            _,
        ) = quantize_activation_top2(
            x, FORMAT, FORMAT.activation_chunk_rows
        )
        self._activation_counts.add_(activation_counts)
        self._forward_calls.add_(1)
        profile_dew_widths(
            x_quantized,
            packed_tags,
            self.linear.weight,
            self._profile_counts,
        )
        return fused_top2_dewa_linear(
            x_quantized,
            packed_tags,
            self.linear.weight,
            self.linear.bias,
            FUNCTIONAL_DEWA,
            self._functional_kernel_stats,
        )

    def export_raw(self):
        rows = self._profile_counts.detach().cpu().tolist()
        return {
            "layer_name": self.layer_name,
            "layer_type": self.layer_name.rsplit(".", 1)[-1],
            "transformer_layer": int(
                re.search(r"model\.layers\.(\d+)", self.layer_name).group(1)
            ),
            "in_features": self.linear.in_features,
            "out_features": self.linear.out_features,
            "forward_calls": int(self._forward_calls.detach().cpu()),
            "weight_counts": self.weight_counts,
            "activation_counts": _counts_to_dict(
                self._activation_counts, ACTIVATION_COUNT_NAMES
            ),
            "functional_stream_kernel_counts": _named_stats(
                self._functional_kernel_stats, KERNEL_STAT_NAMES
            ),
            "profile_by_width": [
                {
                    "acc_width": width,
                    "counts": {
                        name: int(value)
                        for name, value in zip(PROFILE_STAT_NAMES, row)
                    },
                }
                for width, row in zip(ACC_WIDTHS, rows)
            ],
        }


def replace_profiled_linear_layers(module, prefix="model"):
    replaced = {}
    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name
        if isinstance(child, nn.Linear):
            if child.in_features % FORMAT.block_size:
                raise ValueError(f"{full_name}: K must be divisible by G16.")
            weight_counts = quantize_weight_in_place(child.weight, FORMAT)
            wrapper = ProfiledTop2DEWLinear(
                child, full_name, weight_counts
            )
            setattr(module, name, wrapper)
            replaced[full_name] = wrapper
        else:
            replaced.update(
                replace_profiled_linear_layers(child, full_name)
            )
    return replaced


def _rate(numerator, denominator):
    numerator = int(numerator)
    denominator = int(denominator)
    return {
        "numerator": numerator,
        "denominator": denominator,
        "rate": None if denominator == 0 else numerator / denominator,
    }


def _empty_host_counts():
    return {name: 0 for name in PROFILE_STAT_NAMES}


def _add_host_counts(destination, source):
    for name in PROFILE_STAT_NAMES:
        destination[name] += int(source[name])


def _summarize_profile_counts(width, counts):
    return {
        "acc_width": int(width),
        "counts": {name: int(counts[name]) for name in PROFILE_STAT_NAMES},
        "rates": {
            "overflowed_output_rate": _rate(
                counts["overflowed_output_count"],
                counts["total_output_count"],
            ),
            "overflow_event_rate_per_align": _rate(
                counts["overflow_event_count"],
                counts["normal_align_attempt_count"],
            ),
            "overflow_traffic_rate_per_group_slot": _rate(
                counts["overflow_event_count"],
                counts["total_group_output_slots"],
            ),
        },
    }


def aggregate_overflow_results(replaced):
    layer_rows = [replaced[name].export_raw() for name in sorted(replaced)]
    global_counts = [_empty_host_counts() for _ in ACC_WIDTHS]
    type_counts = {}
    transformer_counts = {}

    for layer in layer_rows:
        layer_type = layer["layer_type"]
        transformer_layer = str(layer["transformer_layer"])
        type_counts.setdefault(
            layer_type, [_empty_host_counts() for _ in ACC_WIDTHS]
        )
        transformer_counts.setdefault(
            transformer_layer, [_empty_host_counts() for _ in ACC_WIDTHS]
        )
        for index, width_row in enumerate(layer["profile_by_width"]):
            counts = width_row["counts"]
            _add_host_counts(global_counts[index], counts)
            _add_host_counts(type_counts[layer_type][index], counts)
            _add_host_counts(
                transformer_counts[transformer_layer][index], counts
            )
            layer["profile_by_width"][index] = _summarize_profile_counts(
                ACC_WIDTHS[index], counts
            )

    return {
        "global_by_width": [
            _summarize_profile_counts(width, counts)
            for width, counts in zip(ACC_WIDTHS, global_counts)
        ],
        "layer_type_by_width": {
            layer_type: [
                _summarize_profile_counts(width, counts)
                for width, counts in zip(ACC_WIDTHS, rows)
            ]
            for layer_type, rows in sorted(type_counts.items())
        },
        "transformer_layer_by_width": {
            layer_index: [
                _summarize_profile_counts(width, counts)
                for width, counts in zip(ACC_WIDTHS, rows)
            ]
            for layer_index, rows in sorted(
                transformer_counts.items(), key=lambda item: int(item[0])
            )
        },
        "per_linear_layer": layer_rows,
    }


def validate_aggregates(profile, context_metrics, replaced):
    errors = []
    if len(replaced) != EXPECTED_LINEAR_LAYERS:
        errors.append(
            f"expected {EXPECTED_LINEAR_LAYERS} Linear layers, got {len(replaced)}"
        )
    if context_metrics["evaluated_contexts"] != PROFILE_CONTEXTS:
        errors.append("unexpected evaluated context count")
    if context_metrics["equivalent_causal_loss_tokens"] != (
        PROFILE_CONTEXTS * (CONTEXT_LENGTH - 1)
    ):
        errors.append("unexpected equivalent causal loss token count")

    weight_values = sum(
        layer["weight_counts"]["total_values"]
        for layer in profile["per_linear_layer"]
    )
    activation_values = sum(
        layer["activation_counts"]["total_values"]
        for layer in profile["per_linear_layer"]
    )
    if weight_values != EXPECTED_WEIGHT_VALUES:
        errors.append("unexpected quantized weight value count")
    expected_activation_values = (
        EXPECTED_ACTIVATION_VALUES
        // EXPECTED_CONTEXTS
        * PROFILE_CONTEXTS
    )
    if activation_values != expected_activation_values:
        errors.append("unexpected quantized activation value count")

    previous_overflowed_outputs = None
    for width_row in profile["global_by_width"]:
        counts = width_row["counts"]
        partition = (
            counts["normal_zero_slots"]
            + counts["normal_load_count"]
            + counts["normal_skip_count"]
            + counts["normal_replace_count"]
            + counts["normal_align_attempt_count"]
        )
        if partition != counts["total_group_output_slots"]:
            errors.append(
                f"W{width_row['acc_width']} group-slot partition failed"
            )
        if counts["overflow_event_count"] > counts["normal_align_attempt_count"]:
            errors.append(
                f"W{width_row['acc_width']} overflow exceeds align count"
            )
        if counts["overflowed_output_count"] > counts["total_output_count"]:
            errors.append(
                f"W{width_row['acc_width']} overflowed output closure failed"
            )
        current = counts["overflowed_output_count"]
        if previous_overflowed_outputs is not None and current > previous_overflowed_outputs:
            errors.append("overflowed-output count is not monotonic by width")
        previous_overflowed_outputs = current

    return {
        "passed": not errors,
        "errors": errors,
        "expected_linear_layers": EXPECTED_LINEAR_LAYERS,
        "actual_linear_layers": len(replaced),
        "expected_contexts": PROFILE_CONTEXTS,
        "actual_contexts": context_metrics["evaluated_contexts"],
        "weight_values": weight_values,
        "activation_values": activation_values,
        "expected_activation_values": expected_activation_values,
    }


## Synthetic reference and PyTorch/Triton parity tests


In [ ]:
# RTL state-machine boundary tests.
_partials = torch.tensor(
    [700, 700, 647, 1, 100], dtype=torch.int64
).reshape(-1, 1, 1)
_exponents = torch.zeros_like(_partials)
_counts, _acc, _, _valid, _overflowed = _run_dew_width_reference(
    _partials, _exponents, 12
)
_named = dict(zip(PROFILE_STAT_NAMES, _counts.tolist()))
assert _named["overflow_event_count"] == 1
assert _named["overflowed_output_count"] == 1
assert _named["normal_load_count"] == 2
assert _acc.item() == 100 and _valid.item() and _overflowed.item()

_negative = torch.tensor(
    [-700, -700, -648, -1], dtype=torch.int64
).reshape(-1, 1, 1)
_counts, _, _, _, _ = _run_dew_width_reference(
    _negative, torch.zeros_like(_negative), 12
)
assert _counts[6].item() == 1

_policy_partial = torch.tensor([1, 1, 1], dtype=torch.int64).reshape(-1, 1, 1)
_policy_exp = torch.tensor([10, 1, 13], dtype=torch.int64).reshape(-1, 1, 1)
_counts, _, _, _, _ = _run_dew_width_reference(
    _policy_partial, _policy_exp, 24
)
assert _counts[3].item() == 1
assert _counts[4].item() == 1

_shift_partial = torch.tensor([-5, 1], dtype=torch.int64).reshape(-1, 1, 1)
_shift_exp = torch.tensor([0, 1], dtype=torch.int64).reshape(-1, 1, 1)
_, _shift_acc, _, _, _ = _run_dew_width_reference(
    _shift_partial, _shift_exp, 24
)
assert _shift_acc.item() == -1, "Negative alignment must truncate toward zero."

TRITON_PARITY = {"passed": False, "errors": []}
if torch.cuda.is_available():
    torch.manual_seed(19)
    _m, _n, _k = 5, 17, 64
    _groups = _k // 16
    _x_mantissa = torch.randint(-7, 8, (_m, _groups, 16), device="cuda")
    _w_mantissa = torch.randint(-7, 8, (_n, _groups, 16), device="cuda")
    _x_mantissa[:, :, 0] = 7
    _w_mantissa[:, :, 0] = 7
    _x_exp = torch.randint(-3, 4, (_m, _groups), device="cuda")
    _w_exp = torch.randint(-3, 4, (_n, _groups), device="cuda")
    _x = (
        _x_mantissa.float()
        * torch.pow(2.0, _x_exp.float()).unsqueeze(-1)
    ).reshape(_m, _k).half()
    _w = (
        _w_mantissa.float()
        * torch.pow(2.0, _w_exp.float()).unsqueeze(-1)
    ).reshape(_n, _k).half()
    _tags = torch.zeros((_m, _groups, 16), dtype=torch.bool, device="cuda")
    _tags[:, :, 14] = True
    _tags[::2, :, 15] = True
    _packed = _pack_bits_last_dim(_tags.reshape(_m, _k))
    _partials, _enew = reconstruct_integer_partials(
        _x, _packed, _w
    )
    _reference = profile_dew_reference(_partials, _enew)
    _actual = _empty_profile_counts(torch.device("cuda"))
    profile_dew_widths(_x, _packed, _w, _actual)
    torch.cuda.synchronize()
    if not torch.equal(_actual, _reference):
        TRITON_PARITY["errors"].append(
            {
                "actual": _actual.cpu().tolist(),
                "reference": _reference.cpu().tolist(),
            }
        )
    TRITON_PARITY["passed"] = not TRITON_PARITY["errors"]
    if not TRITON_PARITY["passed"]:
        raise RuntimeError("PyTorch/Triton profiler parity failed.")
    del (
        _x_mantissa,
        _w_mantissa,
        _x_exp,
        _w_exp,
        _x,
        _w,
        _tags,
        _packed,
        _partials,
        _enew,
        _reference,
        _actual,
    )
    torch.cuda.empty_cache()
else:
    TRITON_PARITY["errors"].append("CUDA unavailable; parity test skipped.")

print("RTL reference unit tests passed.")
print(f"Triton parity: {TRITON_PARITY['passed']}")


## WikiText-2 test workload


In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    raise RuntimeError(
        "HF_TOKEN is unavailable. Add it to Colab Secrets or the environment."
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
input_ids = tokenizer(text, return_tensors="pt").input_ids
usable_length = input_ids.size(1) // CONTEXT_LENGTH * CONTEXT_LENGTH
total_contexts = usable_length // CONTEXT_LENGTH
if total_contexts != EXPECTED_CONTEXTS:
    raise RuntimeError(
        f"Expected {EXPECTED_CONTEXTS} contexts, got {total_contexts}."
    )
print(f"WikiText-2 tokens: {input_ids.numel():,}")
print(f"Complete contexts: {total_contexts}")


## Load LLaMA2-7B and quantize 224 decoder Linear layers


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Full profiling requires an NVIDIA CUDA GPU.")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map=0,
    attn_implementation="eager",
    token=token,
)
model.eval()
model.config.use_cache = False

profiled_layers = replace_profiled_linear_layers(
    model.model, prefix="model"
)
if len(profiled_layers) != EXPECTED_LINEAR_LAYERS:
    raise RuntimeError(
        f"Expected {EXPECTED_LINEAR_LAYERS} profiled Linear layers, "
        f"got {len(profiled_layers)}."
    )
if not isinstance(model.lm_head, nn.Linear):
    raise RuntimeError("lm_head must remain an unwrapped FP16 nn.Linear.")

torch.cuda.empty_cache()
print(f"Profiled decoder Linear layers: {len(profiled_layers)}")
print("lm_head remains FP16 and is not executed by the profiling loop.")


## Configurable-context overflow profiling run with resume


In [ ]:
def _checkpoint_signature():
    return {
        "version": 2,
        "model_id": MODEL_ID,
        "context_length": CONTEXT_LENGTH,
        "profile_contexts": PROFILE_CONTEXTS,
        "acc_widths": list(ACC_WIDTHS),
        "t_skip_bits": T_SKIP_BITS,
        "t_replace_bits": T_REPLACE_BITS,
        "linear_layers": sorted(profiled_layers),
    }


def _write_profile_checkpoint(completed_contexts):
    payload = {
        "signature": _checkpoint_signature(),
        "completed_contexts": int(completed_contexts),
        "layers": {
            name: layer.checkpoint_state()
            for name, layer in profiled_layers.items()
        },
    }
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = CHECKPOINT_PATH.with_suffix(".tmp")
    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle)
    temporary_path.replace(CHECKPOINT_PATH)


def _restore_profile_checkpoint():
    for layer in profiled_layers.values():
        layer.reset_runtime_stats()
    if not RESUME_PROFILE or not CHECKPOINT_PATH.is_file():
        return 0
    with CHECKPOINT_PATH.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if payload.get("signature") != _checkpoint_signature():
        raise RuntimeError(
            "Profile checkpoint configuration mismatch. "
            f"Delete {CHECKPOINT_PATH} before starting this run."
        )
    layer_states = payload.get("layers", {})
    if set(layer_states) != set(profiled_layers):
        raise RuntimeError("Profile checkpoint layer set mismatch.")
    for name, layer in profiled_layers.items():
        layer.restore_checkpoint_state(layer_states[name])
    completed = int(payload["completed_contexts"])
    if not 0 <= completed <= PROFILE_CONTEXTS:
        raise RuntimeError("Invalid completed context count in checkpoint.")
    return completed


@torch.inference_mode()
def profile_all_contexts(model, input_ids):
    device = next(model.parameters()).device
    available_contexts = input_ids.size(1) // CONTEXT_LENGTH
    if PROFILE_CONTEXTS > available_contexts:
        raise RuntimeError("Not enough complete contexts for profiling.")
    start_context = _restore_profile_checkpoint()

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start = time.perf_counter()
    progress = tqdm(
        range(start_context, PROFILE_CONTEXTS),
        total=PROFILE_CONTEXTS,
        initial=start_context,
        desc="LLaMA2-7B DEW-ACC W12-W32 overflow profile",
    )
    for context_index in progress:
        begin = context_index * STRIDE
        batch = input_ids[:, begin : begin + CONTEXT_LENGTH].to(device)
        hidden = model.model(
            input_ids=batch,
            use_cache=False,
            return_dict=False,
        )[0]
        del batch, hidden
        completed = context_index + 1
        if (
            completed % CHECKPOINT_EVERY_CONTEXTS == 0
            or completed == PROFILE_CONTEXTS
        ):
            _write_profile_checkpoint(completed)

    torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start
    executed = PROFILE_CONTEXTS - start_context
    return {
        "evaluated_contexts": PROFILE_CONTEXTS,
        "resumed_from_context": start_context,
        "contexts_executed_this_run": executed,
        "input_tokens_per_context": CONTEXT_LENGTH,
        "evaluated_input_tokens": PROFILE_CONTEXTS * CONTEXT_LENGTH,
        "equivalent_causal_loss_tokens": (
            PROFILE_CONTEXTS * (CONTEXT_LENGTH - 1)
        ),
        "elapsed_seconds_this_run": elapsed,
        "contexts_per_second_this_run": (
            None if executed == 0 else executed / elapsed
        ),
        "peak_gpu_memory_gib": (
            torch.cuda.max_memory_allocated(device) / 2**30
        ),
        "checkpoint_path": str(CHECKPOINT_PATH),
    }


context_metrics = profile_all_contexts(model, input_ids)
overflow_profile = aggregate_overflow_results(profiled_layers)
aggregate_validation = validate_aggregates(
    overflow_profile, context_metrics, profiled_layers
)
if not aggregate_validation["passed"]:
    raise RuntimeError(
        "Aggregate validation failed: "
        + str(aggregate_validation["errors"])
    )
print(json.dumps(context_metrics, indent=2))
print("Full-model aggregate validation passed.")


## Optional Experiment 10 RTL trace cross-check


In [ ]:
def _find_repo_file(relative_path):
    relative_path = Path(relative_path)
    candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/research1")]
    for base in candidates:
        candidate = base / relative_path
        if candidate.is_file():
            return candidate
    return None


TRACE_RELATIVE_PATH = Path(
    "experiments/10_RTL_Trace/llama2-7b/"
    "llama2_layer16_down_proj_g16_wbfp4_t2abie4_dew_tskip9_treplace3/"
    "input.dat"
)


def _unpack_g16(hex_value, bits_per_lane):
    value = int(hex_value, 16)
    mask = (1 << bits_per_lane) - 1
    return [(value >> (lane * bits_per_lane)) & mask for lane in range(16)]


def _signed_products(weight_fields, activation_fields, oi1, oi2):
    w_sign, w_exp, w_man = weight_fields
    a_sign, a_exp, a_man = activation_fields
    w_sign_bits = _unpack_g16(w_sign, 1)
    a_sign_bits = _unpack_g16(a_sign, 1)
    w_mantissas = _unpack_g16(w_man, 3)
    a_mantissas = _unpack_g16(a_man, 3)
    if oi1 == 15 and oi2 == 15:
        outliers = set()
    elif oi2 == 0:
        outliers = {oi1}
    else:
        outliers = {oi1, oi2}
    partial = 0
    for lane in range(16):
        if lane in outliers:
            continue
        product = w_mantissas[lane] * a_mantissas[lane]
        partial += -product if w_sign_bits[lane] ^ a_sign_bits[lane] else product
    return partial, int(w_exp, 16) + int(a_exp, 16) - 15


def validate_experiment10_trace(path):
    global_counts = [
        {name: 0 for name in PROFILE_STAT_NAMES} for _ in ACC_WIDTHS
    ]
    states = None
    current_weight = None
    groups_in_dot = 0
    dot_count = 0
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            fields = line.split()
            if not fields:
                continue
            acc_clear = int(fields[0])
            if acc_clear:
                if states is not None:
                    raise RuntimeError("Trace started a dot before closing the prior dot.")
                current_weight = (fields[3], fields[4], fields[5])
                states = [
                    {"valid": False, "exp": 0, "acc": 0, "overflowed": False}
                    for _ in ACC_WIDTHS
                ]
                groups_in_dot = 0
                continue
            if int(fields[2]) != 1 or states is None:
                raise RuntimeError("Unexpected trace record.")
            partial, enew = _signed_products(
                current_weight,
                (fields[6], fields[7], fields[9]),
                int(fields[10], 16),
                int(fields[11], 16),
            )
            current_weight = (fields[3], fields[4], fields[5])
            groups_in_dot += 1
            for index, width in enumerate(ACC_WIDTHS):
                state = states[index]
                counts = global_counts[index]
                counts["total_group_output_slots"] += 1
                if partial == 0:
                    counts["normal_zero_slots"] += 1
                    continue
                if not state["valid"]:
                    counts["normal_load_count"] += 1
                    state.update(valid=True, exp=enew, acc=partial)
                    continue
                delta = enew - state["exp"]
                if delta <= -T_SKIP_BITS:
                    counts["normal_skip_count"] += 1
                elif delta >= T_REPLACE_BITS:
                    counts["normal_replace_count"] += 1
                    state.update(exp=enew, acc=partial)
                else:
                    counts["normal_align_attempt_count"] += 1
                    if delta < 0:
                        shifted = (abs(partial) >> -delta) * (-1 if partial < 0 else 1)
                        candidate = state["acc"] + shifted
                    else:
                        shifted = (abs(state["acc"]) >> delta) * (-1 if state["acc"] < 0 else 1)
                        candidate = partial + shifted
                    minimum = -(1 << (width - 1))
                    maximum = (1 << (width - 1)) - 1
                    if candidate < minimum or candidate > maximum:
                        counts["overflow_event_count"] += 1
                        state.update(valid=False, exp=0, acc=0, overflowed=True)
                    else:
                        state["acc"] = candidate
                        if delta >= 0:
                            state["exp"] = enew
            if groups_in_dot == 688:
                for index, state in enumerate(states):
                    counts = global_counts[index]
                    counts["total_output_count"] += 1
                    counts["overflowed_output_count"] += int(state["overflowed"])
                    counts["final_nonzero_count"] += int(
                        state["valid"] and state["acc"] != 0
                    )
                dot_count += 1
                states = None
    if states is not None or dot_count != 256:
        raise RuntimeError("Experiment 10 trace closure failed.")
    rows = [
        _summarize_profile_counts(width, counts)
        for width, counts in zip(ACC_WIDTHS, global_counts)
    ]
    w24 = rows[ACC_WIDTHS.index(24)]["counts"]
    errors = []
    if any(row["counts"]["overflow_event_count"] != 0 for row in rows):
        errors.append("expected zero W12-W32 overflow events")
    if w24["normal_skip_count"] != 0:
        errors.append("expected W24 skip count 0")
    if w24["normal_replace_count"] != 6:
        errors.append("expected W24 replace count 6")
    return {
        "status": "passed" if not errors else "failed",
        "path": str(path),
        "dot_products": dot_count,
        "errors": errors,
        "by_width": rows,
    }


trace_path = _find_repo_file(TRACE_RELATIVE_PATH)
if trace_path is None:
    TRACE_VALIDATION = {
        "status": "skipped_missing_trace",
        "expected_relative_path": str(TRACE_RELATIVE_PATH),
    }
else:
    TRACE_VALIDATION = validate_experiment10_trace(trace_path)
    if TRACE_VALIDATION["status"] != "passed":
        raise RuntimeError(
            "Experiment 10 trace validation failed: "
            + str(TRACE_VALIDATION["errors"])
        )
print(f"Experiment 10 trace validation: {TRACE_VALIDATION['status']}")


## Export the single JSON artifact


In [ ]:
if set(map(int, EMBEDDED_AREA_BY_WIDTH)) != set(ACC_WIDTHS):
    raise RuntimeError("Embedded Area snapshot does not cover W12-W32.")

result = {
    "schema_version": 1,
    "experiment": {
        "name": "LLaMA2-7B DEW-ACC bit-width overflow profile",
        "model": MODEL_ID,
        "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
        "split": SPLIT,
        "context_length": CONTEXT_LENGTH,
        "stride": STRIDE,
        "drop_remainder": DROP_REMAINDER,
        "quantized_linear_layers": len(profiled_layers),
        "lm_head_profiled": False,
        "activation_stream": (
            "single frozen Top-2 T2A-BiE4 stream generated by the "
            "sticky-Emax functional path; widths do not alter model output"
        ),
    },
    "quantization": {
        "weight": "W-BFP4 G16 signed E5",
        "activation": "Top-2-capped T2A-BiE4 G16 dual signed E5",
        "threshold": "mean(X) + 3 * population_std(X)",
        "candidate_rule": "abs(x) > threshold",
        "top2_tie_break": "lowest K/lane index",
        "rounding": "nearest_even",
    },
    "dew_acc_contract": {
        "widths": list(ACC_WIDTHS),
        "t_skip": T_SKIP_BITS,
        "t_replace": T_REPLACE_BITS,
        "enew": "Ew + Ea_normal",
        "psum_lod": False,
        "alignment_shift": "smaller-exponent signed magnitude toward zero",
        "overflow_detection": "signed W-bit range on ALIGN candidate",
        "overflow_action": "count event and clear accumulator state",
    },
    "widths": list(ACC_WIDTHS),
    "global_by_width": overflow_profile["global_by_width"],
    "layer_type_by_width": overflow_profile["layer_type_by_width"],
    "transformer_layer_by_width": overflow_profile[
        "transformer_layer_by_width"
    ],
    "per_linear_layer": overflow_profile["per_linear_layer"],
    "area_by_width": {
        width: EMBEDDED_AREA_BY_WIDTH[str(width)] for width in ACC_WIDTHS
    },
    "validation": {
        "rtl_reference_unit_tests": {"passed": True},
        "pytorch_triton_parity": TRITON_PARITY,
        "experiment10_trace": TRACE_VALIDATION,
        "full_model_aggregate": aggregate_validation,
    },
    "runtime": {
        **context_metrics,
        "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda,
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "triton": triton.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
    },
}

RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULT_PATH.write_text(
    json.dumps(result, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
reloaded = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
if reloaded["schema_version"] != 1:
    raise RuntimeError("JSON reload validation failed.")

summary = {
    "result_path": str(RESULT_PATH.resolve()),
    "widths": [ACC_WIDTHS[0], ACC_WIDTHS[-1]],
    "linear_layers": len(profiled_layers),
    "contexts": context_metrics["evaluated_contexts"],
    "validation_passed": aggregate_validation["passed"],
    "w12_overflowed_output_rate": result["global_by_width"][0]["rates"][
        "overflowed_output_rate"
    ],
    "w32_overflowed_output_rate": result["global_by_width"][-1]["rates"][
        "overflowed_output_rate"
    ],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


## Download the same JSON artifact


In [ ]:
if not RESULT_PATH.is_file():
    raise FileNotFoundError(f"Result JSON does not exist: {RESULT_PATH}")
try:
    from google.colab import files
except ImportError:
    print(f"JSON remains at: {RESULT_PATH.resolve()}")
else:
    files.download(str(RESULT_PATH))
